In [3]:
# Cell 1 — Imports + paths

import pandas as pd
import numpy as np
import math
from pathlib import Path

DATA_POSTS_PATH = "../data/Ambivalent_Data_Final_2.csv"
DATA_COMMENTS_PATH = "../data/Ambivalent_Comments_Final.csv"

OUT_PREVIEW_PATH = "Ambivalent_Binning_preview_5posts.csv"
OUT_FULL_PATH = "Ambivalent_Binning.csv"


In [8]:
# Cell 2 — Load + filter to posts that have comments

posts = pd.read_csv(DATA_POSTS_PATH)
comments = pd.read_csv(DATA_COMMENTS_PATH)

posts["post_id"] = posts["post_id"].astype(str)
comments["post_id"] = comments["post_id"].astype(str)

comments["comment_score"] = pd.to_numeric(comments["comment_score"], errors="coerce")
comments = comments.dropna(subset=["comment_score"]).copy()
comments["comment_score"] = comments["comment_score"].astype(float)

# only posts that exist in comments
post_ids_with_comments = set(comments["post_id"].unique())
posts_with_comments = posts[posts["post_id"].isin(post_ids_with_comments)].copy()

print("Posts total:", len(posts))
print("Posts with comments:", len(posts_with_comments))
print("Comments total:", len(comments))


Posts total: 1279
Posts with comments: 1168
Comments total: 5614


In [9]:
# Cell 3 — Freedman–Diaconis edges PER POST

def fd_bin_edges(x: np.ndarray) -> np.ndarray:
    """
    Return bin edges using Freedman–Diaconis rule for array x (1D).
    Uses a fallback if IQR==0 or too-few points.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.array([0.0, 1.0])
    if n == 1:
        return np.array([x[0], x[0] + 1e-9])

    xmin, xmax = float(np.min(x)), float(np.max(x))
    if xmin == xmax:
        return np.array([xmin, xmax + 1e-9])

    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1

    if iqr == 0:
        # fallback width
        std = float(np.std(x))
        width = std * (n ** (-1/3))
        if (not np.isfinite(width)) or width <= 0:
            width = 1.0
    else:
        width = 2 * iqr * (n ** (-1/3))

    k = int(math.ceil((xmax - xmin) / width))
    k = max(k, 1)

    edges = np.linspace(xmin, xmax, k + 1)
    edges[-1] = edges[-1] + 1e-9  # include max in left-closed bins
    return edges


In [10]:
# Cell 4 — Build per-post binned comments (then pad to max bins as columns bin01_comment..binNN_comment)

POST_COLS = [
    "post_id","subreddit","title","author","score","num_comments_listed","op_replied",
    "op_reply_count","created_utc","permalink","body","combined_text"
]

def per_post_binning(posts_subset: pd.DataFrame,
                     comments_df: pd.DataFrame,
                     max_comment_chars: int = 350):
    """
    For each post_id:
      - compute FD bins using that post's comment_scores
      - pick the lowest-score comment per bin
      - order bins by increasing score range
    Returns:
      - out_df wide with bin columns padded to max bins in this subset
      - bin_counts dict {post_id: num_bins_used}
      - max_bins used
    """
    # only comments for these posts
    ids = posts_subset["post_id"].astype(str).unique()
    c = comments_df[comments_df["post_id"].isin(ids)].copy()

    # Precompute chosen comments per post as a list (in bin order)
    chosen_map = {}
    bin_counts = {}

    for pid, g in c.groupby("post_id"):
        scores = g["comment_score"].to_numpy()
        edges = fd_bin_edges(scores)

        # assign bin index within this post
        bin_idx = pd.cut(
            g["comment_score"],
            bins=edges,
            right=False,
            include_lowest=True,
            labels=False
        )
        gg = g.copy()
        gg["bin_idx"] = bin_idx
        gg = gg.dropna(subset=["bin_idx"]).copy()
        gg["bin_idx"] = gg["bin_idx"].astype(int)

        # pick min score per bin
        gg = gg.sort_values(["bin_idx", "comment_score"], ascending=[True, True])
        picked = gg.groupby("bin_idx", as_index=False).first()

        # build labels for ranges
        intervals = pd.IntervalIndex.from_breaks(edges, closed="left")
        labels = {i: f"[{intervals[i].left:g}, {intervals[i].right:g})" for i in range(len(intervals))}

        # create ordered list of cell strings
        cells = []
        for _, row in picked.sort_values("bin_idx").iterrows():
            b = int(row["bin_idx"])
            rng = labels.get(b, "")
            score = float(row["comment_score"])
            text = str(row.get("comment_body", "")).replace("\n", " ").replace("\r", " ")
            if len(text) > max_comment_chars:
                text = text[:max_comment_chars].rstrip() + "..."
            cells.append(f"{rng} | score={score:g} | {text}")

        chosen_map[pid] = cells
        bin_counts[pid] = len(cells)

    max_bins = max(bin_counts.values()) if bin_counts else 0

    # build wide rows
    rows = []
    meta = posts_subset[POST_COLS].drop_duplicates(subset=["post_id"]).copy()

    for _, prow in meta.iterrows():
        pid = str(prow["post_id"])
        cells = chosen_map.get(pid, [])
        row = prow.to_dict()
        for j in range(max_bins):
            col = f"bin{j+1:02d}_comment"
            row[col] = cells[j] if j < len(cells) else np.nan
        rows.append(row)

    out_df = pd.DataFrame(rows)
    return out_df, bin_counts, max_bins


In [11]:
# Cell 5 — PREVIEW on first 5 posts (and save). Columns count == max bins among these 5 posts.

preview_posts = posts_with_comments.head(5).copy()
preview_out, preview_bin_counts, preview_max_bins = per_post_binning(preview_posts, comments, max_comment_chars=350)

print("Bin counts per post_id (preview):")
for k, v in preview_bin_counts.items():
    print(k, "->", v)

print("Max bins in preview subset:", preview_max_bins)
print("Preview shape:", preview_out.shape)

display(preview_out.head())

preview_out.to_csv(OUT_PREVIEW_PATH, index=False)
print("Saved preview to:", OUT_PREVIEW_PATH)


Bin counts per post_id (preview):
1bautv8 -> 2
1c09z00 -> 1
1ccnt06 -> 1
1f2kibx -> 2
1hv4zdy -> 3
Max bins in preview subset: 3
Preview shape: (5, 15)


,post_id,subreddit,title,author,score,num_comments_listed,op_replied,op_reply_count,created_utc,permalink,body,combined_text,bin01_comment,bin02_comment,bin03_comment
0,1hv4zdy,meToo,Was this SA?,Ok-Sugar959,5.0,7.0,True,3,1.736186e+09,/r/meToo/comments/1hv4zdy/was_this_sa/,So l've been in a relationship with a girl who...,1hv4zdy meToo Was this SA? Ok-Sugar959 /r/meTo...,"[1, 1.66667) | score=1 | Yeah, it’s tricky bec...","[1.66667, 2.33333) | score=2 | Thank you","[2.33333, 3) | score=3 | Both SA. P.S. Keep a..."
1,1f2kibx,meToo,Dealing with sexual assault trauma / was it SA?,Ok-Air-6389,6.0,3.0,True,1,1.724774e+09,/r/meToo/comments/1f2kibx/dealing_with_sexual_...,I went through a break up recently with my ex ...,1f2kibx meToo Dealing with sexual assault trau...,"[1, 1.5) | score=1 | Thanks for replying and s...","[1.5, 2) | score=2 | Oh dear, that's a tough s...",NaN
2,1ccnt06,meToo,was this sa,FlimsyImplement4042,3.0,3.0,True,1,1.714038e+09,/r/meToo/comments/1ccnt06/was_this_sa/,was this sa?\n\n\nwhen i was 7 or 8 i had my b...,1ccnt06 meToo was this sa FlimsyImplement4042 ...,"[1, 1) | score=1 | &gt;made me go in the close...",NaN,NaN
3,1c09z00,meToo,Was I sexually assaulted by my ex?,[deleted],3.0,1.0,False,0,1.712714e+09,/r/meToo/comments/1c09z00/was_i_sexually_assau...,"\nAbout seven months ago, I had been seeing so...",1c09z00 meToo Was I sexually assaulted by my e...,"[3, 3) | score=3 | Yes, it was assault. You sp...",NaN,NaN
4,1bautv8,meToo,Not sure if this counts as SA,annoyingpea,4.0,2.0,False,0,1.710025e+09,/r/meToo/comments/1bautv8/not_sure_if_this_cou...,So yesterday I was at a party and I’ve definit...,1bautv8 meToo Not sure if this counts as SA an...,"[1, 3) | score=1 | I’m so sorry this happened ...","[3, 5) | score=5 | yes, this is assault, a per...",NaN


Saved preview to: Ambivalent_Binning_preview_5posts.csv


In [12]:
# Cell 6 — FULL run (WARNING: number of columns will become max bins across ALL posts)
# If you want to CAP columns to (say) 10 globally, tell me and I’ll adjust.

full_out, full_bin_counts, full_max_bins = per_post_binning(posts_with_comments, comments, max_comment_chars=350)

print("Max bins across ALL posts:", full_max_bins)
print("Full shape:", full_out.shape)

full_out.to_csv(OUT_FULL_PATH, index=False)
print("Saved full output to:", OUT_FULL_PATH)


Max bins across ALL posts: 22
Full shape: (1082, 34)
Saved full output to: Ambivalent_Binning.csv


In [13]:
# Remove "[a, b) | score=x | " prefix from all binXX_comment columns

import pandas as pd
import re

IN_PATH = "Ambivalent_Binning.csv"
OUT_PATH = "Ambivalent_Binning_Clean.csv"

df = pd.read_csv(IN_PATH)

bin_cols = [c for c in df.columns if re.match(r"^bin\d+_comment$", c)]

def strip_prefix(val):
    if pd.isna(val):
        return val
    s = str(val)
    # Split on " | " at most 2 times -> [range, score=..., rest_of_comment]
    parts = s.split(" | ", 2)
    if len(parts) == 3:
        return parts[2]
    return s  # if format is unexpected, leave as-is

for c in bin_cols:
    df[c] = df[c].apply(strip_prefix)

df.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
print("Example (before/after) check on first non-null cell:")
for c in bin_cols:
    sample = df[c].dropna()
    if len(sample) > 0:
        print(c, "->", sample.iloc[0][:120], "...")
        break


Saved: Ambivalent_Binning_Clean.csv
Example (before/after) check on first non-null cell:
bin01_comment -> Yeah, it’s tricky because I swear to god I only heard get once, but she said she said it more than once. ...
